# Conceptos para la validación de la reconstrucción DSA

Este cuaderno resume los conceptos empleados para comparar la matriz de
densidad espectral reconstruida desde EEG crudo con la matriz exportada por el
monitor en el archivo `.f_a`.

La metodología principal conserva la escala física de potencia. Las métricas
MAE, RMSE y bias se calculan en decibelios, sin normalizar independientemente
las matrices mediante z-score. El z-score puede utilizarse como análisis
exploratorio de la forma relativa, pero no para medir la fidelidad absoluta.

## SEF y MEF

Para cada segundo se parte de la potencia espectral **lineal** de todos los
bins de frecuencia. Primero se calcula la potencia acumulada:

$$
C(f_j)=\sum_{i=1}^{j}P(f_i)
$$

y la potencia total:

$$
P_{\mathrm{total}}=\sum_{i=1}^{m}P(f_i)
$$

- **SEF**: primera frecuencia para la que
  \(C(f_j)/P_{\mathrm{total}}\geq 0.95\).
- **MEF**: primera frecuencia para la que
  \(C(f_j)/P_{\mathrm{total}}\geq 0.50\).

Estos cálculos deben realizarse sobre potencia lineal, no sobre valores en dB.
En la reconstrucción seleccionada se utiliza el canal 1 para registros
unilaterales y los canales 1 y 3, de forma independiente, para los hemisferios
izquierdo y derecho de los registros bilaterales.

# Comparabilidad de las matrices

Antes de calcular errores se comprueba que las dos matrices:

1. contienen las mismas frecuencias;
2. están alineadas con la línea temporal del `.spa`;
3. utilizan las mismas posiciones válidas;
4. están expresadas en la misma escala física.

El archivo `.f_a` almacena sus valores con un factor de escala de 100, por lo
que se divide entre 100. La PSD reconstruida, expresada en
\(\mu V^2/Hz\), se integra en bins de 0,5 Hz y se convierte a dB con la
referencia del monitor.

Que las matrices presenten rangos observados diferentes no justifica
normalizarlas por separado. Esa diferencia puede reflejar un offset, una
ganancia o un contraste espectral incorrectos y debe quedar recogida por MAE,
RMSE y bias.

# Métricas principales

Las matrices se convierten en dos vectores utilizando únicamente las celdas
finitas comunes.

## Correlación de Pearson

$$
r =
\frac{\sum_i(A_i-\bar A)(B_i-\bar B)}
{\sqrt{\sum_i(A_i-\bar A)^2}\sqrt{\sum_i(B_i-\bar B)^2}}
$$

Mide la semejanza lineal del patrón espectral. Un valor próximo a 1 indica que
las zonas de mayor y menor intensidad varían de forma parecida, pero no
garantiza que los niveles absolutos coincidan.

## MAE

$$
MAE=\frac{1}{n}\sum_{i=1}^{n}|A_i-B_i|
$$

Representa la diferencia absoluta media. Al calcularse sobre las matrices sin
normalizar, se expresa en dB.

## RMSE

$$
RMSE=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(A_i-B_i)^2}
$$

También se expresa en dB y penaliza con mayor intensidad los errores grandes.
Un RMSE mucho mayor que el MAE sugiere la presencia de discrepancias
localizadas de gran magnitud.

## Bias

En este proyecto se define como:

$$
Bias=\frac{1}{n}\sum_{i=1}^{n}(B_i-A_i)
$$

donde \(A\) es la referencia `.f_a` y \(B\) la reconstrucción. Por tanto:

- bias positivo: la reconstrucción queda, en promedio, por encima del `.f_a`;
- bias negativo: la reconstrucción queda por debajo;
- bias próximo a cero: no existe un desplazamiento medio importante, aunque
  todavía pueden existir errores locales.

In [ ]:
import numpy as np


def calcular_metricas_dsa_db(dsa_fa, dsa_reconstruida):
    # Calcula métricas sobre las celdas finitas comunes, sin z-score.
    referencia = np.asarray(dsa_fa, dtype=float)
    candidata = np.asarray(dsa_reconstruida, dtype=float)

    validas = np.isfinite(referencia) & np.isfinite(candidata)
    a = referencia[validas]
    b = candidata[validas]

    if len(a) == 0:
        raise ValueError("No existen celdas válidas comunes.")

    return {
        "n": len(a),
        "Pearson": float(np.corrcoef(a, b)[0, 1]),
        "MAE_dB": float(np.mean(np.abs(a - b))),
        "RMSE_dB": float(np.sqrt(np.mean((a - b) ** 2))),
        "bias_recon_menos_fa_dB": float(np.mean(b - a)),
    }

## Por qué el z-score no se usa en la metodología principal

El z-score transforma cada matriz mediante:

$$
Z=\frac{D-\mu_D}{\sigma_D}
$$

Si cada matriz se normaliza con su propia media y desviación típica, se
eliminan las diferencias globales de nivel y contraste. Una reconstrucción
sistemáticamente varios dB por encima del `.f_a` podría obtener un MAE
normalizado pequeño.

Además, MAE y RMSE dejan de expresarse en dB y pasan a medirse en desviaciones
típicas. Para dos vectores estandarizados, el RMSE está estrechamente ligado a
Pearson:

$$
RMSE_z \simeq \sqrt{2(1-r)}
$$

Por ello, las métricas normalizadas aportan poca información adicional sobre
la fidelidad física. Pueden conservarse como análisis secundario de forma,
etiquetadas explícitamente como `MAE_z` y `RMSE_z`, pero no se utilizan para
seleccionar el método espectral, el suavizado, la máscara o el shift.

## Spearman y análisis locales

La correlación de Spearman evalúa si el orden relativo de las intensidades se
conserva, aunque la relación no sea lineal. Es útil como análisis
complementario, pero no sustituye a Pearson ni a los errores en dB.

También pueden calcularse:

- **correlación por frecuencia**, comparando cada columna temporal;
- **correlación por tiempo**, comparando los vectores espectrales de cada
  segundo.

Estos análisis ayudan a localizar bandas o intervalos problemáticos. Las
métricas globales siguen siendo necesarias para resumir la reconstrucción.

# Suavizado temporal

El campo `SpSmooth` del `.spa` describe la atenuación temporal aplicada al
espectro. En los registros estudiados corresponde a 30 s.

La configuración seleccionada utiliza una media móvil causal:

```python
trabajo = dsa_eeg.copy()
trabajo.loc[mask_base, :] = np.nan

dsa_suavizada = trabajo.rolling(
    window=SpSmooth,
    min_periods=1,
    center=False,
).mean()
```

La máscara se introduce antes del suavizado para que los segundos inválidos no
participen en la media. Pandas ignora los `NaN`, por lo que cada valor se
calcula únicamente con los segundos válidos disponibles dentro de la ventana:

$$
\widetilde D(t,f)=
\frac{\sum_{k=0}^{N-1}I_{\mathrm{val}}(t-k)D(t-k,f)}
{\sum_{k=0}^{N-1}I_{\mathrm{val}}(t-k)}
$$

donde \(I_{\mathrm{val}}\) vale 1 para una observación válida y 0 para una
inválida. Si no existe ninguna observación válida, el resultado permanece como
`NaN`.

Después del suavizado y del shift, la máscara se aplica nuevamente para
conservar las bandas blancas en los instantes originales de pérdida de
información.

## `rolling` frente a `ewm`

`rolling` asigna el mismo peso a todas las observaciones válidas incluidas en
la ventana. `ewm` asigna más peso a los valores recientes y conserva una
memoria exponencial de los anteriores.

En los registros evaluados:

| Suavizado | Pearson medio | MAE (dB) | RMSE (dB) |
|:---|---:|---:|---:|
| `rolling` | **0,9149** | **1,816** | 2,581 |
| `ewm` | 0,9088 | 1,844 | **2,567** |

La diferencia de RMSE a favor de `ewm` fue de solo 0,014 dB. `rolling` obtuvo
mejor Pearson y MAE, por lo que se seleccionó como compromiso global. No debe
afirmarse que ganó todas las métricas ni que `ewm` sea incorrecto.

# Shift temporal

El shift es una prueba de alineación: no cambia la potencia ni las
frecuencias, sino qué segundo de la reconstrucción se compara con cada segundo
del `.f_a`.

Para un shift positivo \(s\):

```python
referencia = dsa_fa.iloc[s:].reset_index(drop=True)
candidata = dsa_suavizada.iloc[:-s].reset_index(drop=True)
```

La candidata calculada a partir de instantes anteriores se compara con una
referencia situada \(s\) segundos después. En los registros disponibles, los
valores seleccionados fueron 10 s para unilateral y 6 s para bilateral.

En la aplicación se mantiene la duración original creando una matriz llena de
`NaN` y desplazando los valores dentro de ella. La máscara y las variables
procesadas deben permanecer referidas a la línea temporal de destino. El shift
es un ajuste empírico de sincronización y no debe interpretarse como una
modificación del EEG.

# Comparación de técnicas y decisión final

## Welch y `spectrogram`

Con la misma ventana Hann de 2 s, solapamiento de 1 s, detrend y escalado,
ambos produjeron exactamente la misma PSD y las mismas métricas. En estas
condiciones, `spectrogram` no es una alternativa metodológica a Welch, sino una
forma distinta de organizar los mismos periodogramas modificados.

## Wavelets

La CWT de Morlet se evaluó con varios anchos de banda. La mejor configuración
según Pearson obtuvo 0,9077, MAE de 3,227 dB y RMSE de 3,833 dB. Welch obtuvo
0,9149, 1,816 dB y 2,581 dB, respectivamente. Las wavelets se descartaron para
la reconstrucción final porque no mejoraron la semejanza con el `.f_a`; esto
no implica que sean inadecuadas para otros objetivos, como detectar
transitorios.

## Máscaras

La máscara base eliminó, en promedio, el 2,9 % de los segundos. ARTF2 eliminó
el 33,6 % y llegó al 70,1 % en el registro unilateral, sin aportar una mejora
consistente. Por ello se conserva la máscara base y ARTF2 queda como análisis
de sensibilidad.

## Configuración seleccionada

- Welch con ventana Hann de 2 s y avance de 1 s.
- Canal 1 en unilateral y canales 1 y 3 en bilateral.
- Filtro pasa-altos causal según `LoFilter`.
- Bins de 0,5 Hz entre 0,5 y 30 Hz.
- Media móvil causal `rolling` según `SpSmooth`.
- Máscara base antes del suavizado y reaplicada al final.
- Shift de 10 s en unilateral y 6 s en bilateral.
- Pearson, MAE, RMSE y bias calculados sin z-score.
- Escala visual lineal común de 49 a 94 dB, sin gamma.

La configuración se validará con los nuevos registros unilaterales manteniendo
todos los parámetros fijos.